In [ ]:
#prompt


"""
This project is for examining the datas of my gym session logs, the logs are provided in the .zip file,
in hyothesis_test.py file, it already contains 2 test,

contemplate on a one more different test,
explain the logic of that test step by step

do not use ANOVA
"""


#output


"""
Test 3 — Session-Type Consistency: Does RQI Differ Significantly Across Session Types (A / B / C)?
The question: Do session types (A, B, C) produce systematically different Rep Quality Index (RQI) values, or is any observed difference explainable by random variation?
Why it's different from Tests 1 & 2:

Tests 1 and 2 are both Pearson correlation tests (linear, continuous × continuous)
This test is a Wilcoxon rank-sum test (non-parametric, comparing distributions between groups)
It operates on grouped categorical structure (session type) rather than a continuous predictor
It makes no normality assumption — appropriate since RQI is a bounded ratio [0,1] that is unlikely to be normally distributed

The logic, step by step:

Compute RQI per session per exercise: RQI = last_set_reps / first_set_reps. This measures how well you maintained performance across sets — 1.0 means no drop, lower means fatigue set in.
Average RQI across all exercises within each session to get one number per session.
Split sessions into groups by type (A, B, C).
Pick the two most populated groups to compare (ensures statistical power). Typically A vs B, or whichever two have the most sessions.
Run Wilcoxon rank-sum test (also known as Mann-Whitney U): rank all RQI values together, then check whether the two groups' ranks are distributed differently than chance would predict.
Two-tailed, because we don't presume which session type is harder — A might be push-dominant, B might be pull-dominant, either could produce lower RQI.
"""


In [ ]:
#prompt


"""

i have a folder called workouts2
in this folder,
i have my fitness logs,
the name of these logs starts from "W-168-C.txt" to "W-404-C.txt",
it does not mean that 404-168 logs exist, the numbers correpond to that day, 
first sample sample from "W-169-A.txt" =>
21Jan2023
bw:
w 0 kg;
kcal:
e 0 kcal;
bench_press:
w 72.5 kg; rps 5s5r;
db_decline_press:
w 20 kg; rps 5s10r;
db_incline_bench_press:
w 20 kg; rps 2s12r;
w 20 kg; rps 3s10r;
cable_fly:
w 12.5 kg; rps 5s10;
db_tri_ex:
w 22.5 kg; rps 3s10r;
one_arm_cable_tri_ex:
w 7.5 kg; rps 4s18r;
second sample from "W-182-B.txt" =>
3Feb2023
bw:
w 0 kg;
kcal:
e 0 kcal;
db_shoulder_press:
w 20 kg; rps 4s10r;
w 20 kg; rps 1s8r;
ez_front_row:
w 25 kg; rps 4s12r;
w 25 kg; rps 1s10r;
lat_raises:
w 15 kg; rps 5s12r;
leg_extension:
w 70 kg; rps 5s12r;
leg_curl:
w 70 kg; rps 1s8r;
w 70 kg; rps 4s6r;
leg_press_calf_raises:
w 160 kg; rps 5s14r;
treadmill:
mins 15;
third sample from "W-232-C.txt" =>
2023-03-25
pull_up:
rps 8,8,7,6,6;
kroc_row:
w 35kg; rps 5s8r;
ez_bar_curl:
w 25kg; rps 10,10,8,8,8;
db_hammer_curl:
w 20kg; rps 3s24r;
one_arm_cable_pulldown:
w 32.25kg; rps 10,10,9,8,8;
one_arm_cable_seated_row:
w 28.25kg; rps 5s10r;
as it can seen, some time later i changed the date structure, u should handle these,
also there could be new exercise names, u should also handle those too 
now create a parsing algorithm that parses these datas and loads them to a dictionary in a meaningful matter
"""






#output


'''
"""
workout_parser.py
-----------------
Parses fitness log files (W-NNN-X.txt) into a structured dictionary.

Output schema
─────────────
{
  "W-169-A": {
    "filename": "W-169-A.txt",
    "day_number": 169,
    "session_type": "A",
    "date": datetime(2023, 1, 21),
    "date_str": "2023-01-21",
    "body_weight_kg": None,          # float or None
    "kcal_eaten": None,              # float or None
    "exercises": [
      {
        "name": "bench_press",
        "sets": [
          {
            "weight_kg": 72.5,       # float or None (None when bodyweight)
            "bodyweight": False,     # True when weight is 'bw'
            "reps": [5, 5, 5, 5, 5], # list[int]
          },
          ...
        ],
        "cardio_minutes": None,      # int, only for treadmill-style entries
      },
      ...
    ],
    "parse_warnings": [],            # list of strings for anything fuzzy
  },
  ...
}
"""

import os
import re
from datetime import datetime
from pathlib import Path
from pprint import pformat


# ─────────────────────────────────────────────────────────────────────────────
# Date parsing
# ─────────────────────────────────────────────────────────────────────────────

_DATE_FORMATS = [
    # "21Jan2023"  /  "3Feb2023"
    ("%d%b%Y", re.compile(r"^\d{1,2}[A-Za-z]{3}\d{4}$")),
    # "2023-03-25"
    ("%Y-%m-%d", re.compile(r"^\d{4}-\d{2}-\d{2}$")),
    # "21/01/2023"  (just in case)
    ("%d/%m/%Y", re.compile(r"^\d{1,2}/\d{1,2}/\d{4}$")),
]


def _parse_date(token: str):
    """Return (datetime, date_str) or (None, None) if unrecognised."""
    t = token.strip()
    for fmt, pat in _DATE_FORMATS:
        if pat.match(t):
            try:
                dt = datetime.strptime(t, fmt)
                return dt, dt.strftime("%Y-%m-%d")
            except ValueError:
                pass
    return None, None


# ─────────────────────────────────────────────────────────────────────────────
# Weight parsing
# ─────────────────────────────────────────────────────────────────────────────

def _parse_weight(raw: str):
    """
    Parse a weight token like '72.5 kg', '72.5kg', 'bw', '0 kg' …
    Returns (weight_kg: float | None, is_bodyweight: bool).
    weight_kg is None only when is_bodyweight is True.
    """
    raw = raw.strip().lower()
    if raw in ("bw", "bodyweight"):
        return None, True
    # strip unit, allow optional space
    m = re.match(r"([\d.]+)\s*kg?", raw)
    if m:
        return float(m.group(1)), False
    return None, False


# ─────────────────────────────────────────────────────────────────────────────
# Reps parsing
# ─────────────────────────────────────────────────────────────────────────────

def _parse_reps(raw: str):
    """
    Parse a reps token.  Supported forms:
      "5s5r"      → 5 sets of 5 reps  → [5,5,5,5,5]
      "5s10r"     → [10,10,10,10,10]
      "3s24r"     → [24,24,24]
      "8,8,7,6,6" → [8,8,7,6,6]
      "10,10,8,8,8" → [10,10,8,8,8]
    Returns list[int] or [] on failure.
    """
    raw = raw.strip().rstrip("r").rstrip(";")

    # "NsNr" or "NsN"
    m = re.match(r"(\d+)s(\d+)r?$", raw)
    if m:
        sets, reps = int(m.group(1)), int(m.group(2))
        return [reps] * sets

    # comma-separated  "8,8,7,6,6"
    if re.match(r"[\d,]+$", raw):
        return [int(x) for x in raw.split(",") if x]

    return []


# ─────────────────────────────────────────────────────────────────────────────
# Exercise-section line parsers
# ─────────────────────────────────────────────────────────────────────────────

def _parse_set_line(line: str):
    """
    Parse a data line inside an exercise block.
    Examples
      "w 72.5 kg; rps 5s5r;"
      "w 20 kg; rps 2s12r;"
      "w 35kg; rps 5s8r;"
      "w 72.5 kg; rps; 1s5r;"     ← extra semicolon after rps
      "w 200kg; 1s6r;"             ← missing 'rps' keyword
      "w 0 kg;"                    ← weight-only (bw block)
      "rps 8,8,7,6,6;"
      "mins 15;"
    Returns one of:
      ("set",     {weight_kg, bodyweight, reps})
      ("cardio",  {minutes})
      ("kcal",    {kcal})
      ("unknown", raw_line)
    """
    line = line.rstrip(";").strip()

    # cardio / treadmill
    m = re.match(r"mins\s+(\d+)", line, re.IGNORECASE)
    if m:
        return "cardio", {"minutes": int(m.group(1))}

    # energy  "e 0 kcal"
    m = re.match(r"e\s+([\d.]+)\s*kcal", line, re.IGNORECASE)
    if m:
        return "kcal", {"kcal": float(m.group(1))}

    # weight + reps  "w <weight>; rps <reps>"  (possibly "rps; <reps>")
    # Also handles "w <weight>; <reps>" (missing rps keyword)
    m = re.match(r"w\s+(.+?);\s*(?:rps;?\s*)(.+)", line, re.IGNORECASE)
    if m:
        weight_kg, is_bw = _parse_weight(m.group(1))
        reps = _parse_reps(m.group(2))
        return "set", {"weight_kg": weight_kg, "bodyweight": is_bw, "reps": reps}

    # weight only, no reps  "w 0 kg"  (used in bw block with 0 kg)
    m = re.match(r"^w\s+(.+)$", line, re.IGNORECASE)
    if m:
        weight_kg, is_bw = _parse_weight(m.group(1))
        return "set", {"weight_kg": weight_kg, "bodyweight": is_bw, "reps": []}

    # reps only  "rps 8,8,7,6,6"
    m = re.match(r"rps\s+(.+)", line, re.IGNORECASE)
    if m:
        reps = _parse_reps(m.group(1))
        return "set", {"weight_kg": None, "bodyweight": True, "reps": reps}

    return "unknown", line


def _normalise_exercise_name(raw: str) -> str:
    """Strip trailing colon/semicolons and lower-case the exercise name."""
    return raw.strip().rstrip(":;").strip().lower()


# ─────────────────────────────────────────────────────────────────────────────
# Main file parser
# ─────────────────────────────────────────────────────────────────────────────

_FILENAME_RE = re.compile(r"^W-(\d+)-([A-Za-z])\.txt$", re.IGNORECASE)

# These are meta-keys, not exercises
_META_KEYS = {"bw", "kcal"}


def parse_workout_file(filepath: str | Path) -> dict:
    """Parse one workout log file and return its structured dict."""
    path = Path(filepath)
    fname = path.name

    # Extract day number and session type from filename
    m = _FILENAME_RE.match(fname)
    day_number = int(m.group(1)) if m else None
    session_type = m.group(2).upper() if m else None

    warnings = []

    with open(path, encoding="utf-8", errors="replace") as fh:
        raw_lines = fh.readlines()

    lines = [l.rstrip("\n") for l in raw_lines]

    # ── Pass 1: find the date (first non-empty line that looks like a date) ──
    date_obj, date_str = None, None
    date_line_idx = None
    for i, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue
        date_obj, date_str = _parse_date(stripped)
        if date_obj:
            date_line_idx = i
            break
        # first non-empty line that is NOT a date → warn and move on
        warnings.append(f"Could not parse date from first non-empty line: {repr(stripped)}")
        break

    # ── Pass 2: walk remaining lines to collect exercise blocks ──
    body_weight_kg: float | None = None
    kcal_eaten: float | None = None
    exercises: list[dict] = []

    current_exercise: str | None = None
    current_sets: list[dict] = []
    current_cardio: int | None = None

    def flush_exercise():
        nonlocal current_exercise, current_sets, current_cardio
        if current_exercise is None:
            return
        if current_exercise not in _META_KEYS:
            # Merge rows that share the same weight key into one entry,
            # preserving the original ordering of first appearance.
            merged = {}
            for s in current_sets:
                key = (s["weight_kg"], s["bodyweight"])
                if key in merged:
                    merged[key]["reps"].extend(s["reps"])
                else:
                    merged[key] = {
                        "weight_kg": s["weight_kg"],
                        "bodyweight": s["bodyweight"],
                        "reps": list(s["reps"]),
                    }
            exercises.append({
                "name": current_exercise,
                "sets": list(merged.values()),
                "cardio_minutes": current_cardio,
            })
        current_exercise = None
        current_sets = []
        current_cardio = None

    start = (date_line_idx + 1) if date_line_idx is not None else 0

    for line in lines[start:]:
        stripped = line.strip()
        if not stripped:
            continue

        # ── Exercise header line: ends with ":" or ";" (or neither, just the name) ──
        # Heuristic: no leading spaces AND matches exercise-name pattern
        is_header = (
            not line.startswith((" ", "\t"))
            and re.match(r"^[a-z_]+[;:]?\s*$", stripped, re.IGNORECASE)
        )

        if is_header:
            flush_exercise()
            current_exercise = _normalise_exercise_name(stripped)
            continue

        # ── Data line inside a block ──
        if current_exercise is None:
            warnings.append(f"Orphan data line (no exercise header): {repr(stripped)}")
            continue

        kind, parsed = _parse_set_line(stripped)

        if kind == "set":
            if current_exercise == "bw":
                # body weight entry; 0 means not tracked that day
                bw = parsed["weight_kg"]
                body_weight_kg = bw if (bw is not None and bw != 0.0) else body_weight_kg
            elif current_exercise == "kcal":
                pass  # handled via "kcal" kind below
            else:
                current_sets.append(parsed)

        elif kind == "kcal":
            kcal_eaten = parsed["kcal"]

        elif kind == "cardio":
            current_cardio = parsed["minutes"]

        elif kind == "unknown":
            # Could be "e 0 kcal" under kcal block
            m_e = re.match(r"e\s+([\d.]+)\s*kcal", stripped, re.IGNORECASE)
            if m_e and current_exercise == "kcal":
                kcal_eaten = float(m_e.group(1))
            else:
                warnings.append(f"Unrecognised line under '{current_exercise}': {repr(stripped)}")

    flush_exercise()

    return {
        "filename": fname,
        "day_number": day_number,
        "session_type": session_type,
        "date": date_obj,
        "date_str": date_str,
        "body_weight_kg": body_weight_kg,
        "kcal_eaten": kcal_eaten,
        "exercises": exercises,
        "parse_warnings": warnings,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Directory loader
# ─────────────────────────────────────────────────────────────────────────────

def load_workout_directory(directory: str | Path) -> dict[str, dict]:
    """
    Parse all W-NNN-X.txt files in *directory*.
    Returns an OrderedDict keyed by "W-NNN-X" (filename without extension),
    sorted by day number then session type.
    """
    directory = Path(directory)
    files = sorted(
        directory.glob("W-*-*.txt"),
        key=lambda p: (
            int(m.group(1)) if (m := _FILENAME_RE.match(p.name)) else 9999,
            m.group(2).upper() if m else "Z",
        ),
    )

    result = {}
    for f in files:
        key = f.stem  # "W-169-A"
        result[key] = parse_workout_file(f)

    return result


# ─────────────────────────────────────────────────────────────────────────────
# Quick sanity check / demo
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    WORKOUT_DIR = Path("/home/claude/workouts2/workouts2")
    data = load_workout_directory(WORKOUT_DIR)

    print(f"Loaded {len(data)} workout sessions.\n")

    # Show first 3 sessions
    for key in list(data.keys())[:3]:
        w = data[key]
        print(f"{'─'*60}")
        print(f"Key        : {key}")
        print(f"Date       : {w['date_str']}")
        print(f"Day #      : {w['day_number']}  Session: {w['session_type']}")
        print(f"Body weight: {w['body_weight_kg']} kg")
        print(f"kcal eaten : {w['kcal_eaten']}")
        print(f"Exercises  : {len(w['exercises'])}")
        for ex in w["exercises"]:
            print(f"  [{ex['name']}]")
            if ex["cardio_minutes"]:
                print(f"    cardio: {ex['cardio_minutes']} min")
            for s in ex["sets"]:
                wt = "bw" if s["bodyweight"] else f"{s['weight_kg']} kg"
                print(f"    {wt} × {s['reps']}")
        if w["parse_warnings"]:
            print(f"  ⚠ Warnings: {w['parse_warnings']}")
        print()

    # Aggregate: collect all unique exercise names
    all_exercises = set()
    for w in data.values():
        for ex in w["exercises"]:
            all_exercises.add(ex["name"])

    print(f"\nUnique exercises found ({len(all_exercises)}):")
    for name in sorted(all_exercises):
        print(f"  {name}")

    # Report files with parse warnings
    warned = [(k, v["parse_warnings"]) for k, v in data.items() if v["parse_warnings"]]
    print(f"\nFiles with parse warnings: {len(warned)}")
    for k, w in warned:
        print(f"  {k}: {w}")

'''

In [ ]:
#prompt 

"""
if you look at the W_168_C

you can see that the exercise pullup contains two rows that have the same weight(for this case it is bw),
update the algorithm in such a way that if there are rows with the same weight under a particular exercise, simply concatenate the sets and show it as one row inside the dictionary

"""




#output

"""
pull_up in W-168-C now comes out as a single row: bw × [9, 9, 9, 8, 8] — the two bw lines are merged.
The logic lives inside flush_exercise(): when the exercise block is closed, it walks the collected sets and groups them by (weight_kg, bodyweight) as the key. If the same weight appears again, the new reps are just appended to the existing list. The order of first appearance is preserved, so heavier drop-sets or warmup progressions that use different weights stay in the right sequence.

"""

In [ ]:
#prompt


"""

data collection, EDA(exploratary data analysis), and hypothesis testing

now data collection is done,
do the EDA, Plot your total volume per week over time, create heatmaps and so on(do some meaningful stuff)

create a hypothesis like does increasing the frequency of going to gym also icnreases the strength ?

"""



#output

"""

Here's what the dashboard reveals:
Volume trends. Your weekly volume grew steadily from ~34k kg in January to peaks above 100k kg by May, with a clear plateau/slight dip in summer (June–August) before stabilising around 75–80k kg. Week 29 (a 1-session week) is a visible outlier — probably a deload or travel week.
Heatmap. You trained almost exclusively Monday–Friday, with Saturday appearing occasionally and Sunday almost never. The heaviest training density was February through May — your highest-frequency stretch.
Volume share. Leg press calf raises utterly dominates at 430k kg total — more than double the second-placed seated row (213k kg). This is partly because calf raises use high reps with heavy weight, which inflates volume. Bench press is 8th despite being a primary lift — it uses lower total reps than machine work.
Strength progression. Every tracked lift went up. Bench went 72.5 → 90 kg (+24%), shoulder press 20 → 30 kg (+50%), lat pulldown 63 → 96 kg (+52%), seated row 100 → 160 kg (+60%). The progression is remarkably consistent — almost no regression periods.
Hypothesis result. The hypothesis is strongly supported. The Pearson correlation between sessions/week and weekly volume is r = 0.884, with t = 8.04 (p < 0.0001). Low-frequency weeks (≤3 sessions) averaged 34k kg vs 77k kg for high-frequency weeks — a 2.3× difference. One caveat: this is correlation, not necessarily causation. High-volume weeks are high-volume partly by definition (more sessions = more lifts). A stronger test would be tracking your 1-rep max on a single lift across periods of different frequency — and your bench progression does suggest this holds, as your fastest gains align with your highest-frequency months (Feb–May).

"""


In [ ]:
# prompt
"""
in my code i parse gym logs and try to extract meaningful infos,

similar to the function inside eda,
implement a new function that creates a grap showing strength progression in the exercises of 
bench_press, shoulder_press, leg_extension and lat_pulldown,

for each of them create a different graph,
x-axis showing the timeline, y_axis showing the weight lifted in that timeline

do not use the arrow key symbol when writing comments and do not use the long dash, make comments look natural
"""





# output

'''

# paste this function into eda.py alongside the existing functions


from collections import defaultdict
from datetime import datetime, date
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np


TRACKED_EXERCISES = [
    "bench_press",
    "shoulder_press",
    "leg_extension",
    "lat_pulldown",
]

EXERCISE_COLOR = {
    "bench_press":     "#4C9BE8",
    "shoulder_press":  "#F5A623",
    "leg_extension":   "#57C26A",
    "lat_pulldown":    "#E85D75",
}


def _max_weight_for_exercise(workout: dict, exercise_name: str):
    """
    Find the heaviest non-bodyweight weight used for a given exercise in one workout.
    Returns float or None if the exercise was not done that day.
    """
    for ex in workout["exercises"]:
        if ex["name"] != exercise_name:
            continue
        weights = [
            s["weight_kg"]
            for s in ex["sets"]
            if not s["bodyweight"] and s["weight_kg"] is not None and s["weight_kg"] > 0
        ]
        if weights:
            return max(weights)
    return None


def plot_strength_progression(data: dict, save_path: str | None = None):
    """
    Plot strength progression over time for four exercises:
    bench_press, shoulder_press, leg_extension, lat_pulldown.

    Each exercise gets its own subplot. The x-axis is the workout date
    and the y-axis is the heaviest weight used in that session (kg).
    Bodyweight sets are excluded.

    Parameters
    ----------
    data : dict
        Output of load_workout_directory().
    save_path : str or None
        File path to save the figure as a PNG.
        Pass None to skip saving.

    Returns
    -------
    matplotlib.figure.Figure
    """

    # Build a timeline per exercise: list of (date, max_weight_kg)
    timelines: dict[str, list[tuple[date, float]]] = defaultdict(list)

    for w in data.values():
        if not w["date_str"]:
            continue
        d = datetime.strptime(w["date_str"], "%Y-%m-%d").date()
        for exercise_name in TRACKED_EXERCISES:
            max_w = _max_weight_for_exercise(w, exercise_name)
            if max_w is not None:
                timelines[exercise_name].append((d, max_w))

    # Sort each timeline by date
    for exercise_name in TRACKED_EXERCISES:
        timelines[exercise_name].sort(key=lambda x: x[0])

    fig, axes = plt.subplots(
        2, 2,
        figsize=(16, 10),
        facecolor="white",
    )
    fig.subplots_adjust(hspace=0.45, wspace=0.3)

    axes_flat = axes.flatten()

    for ax, exercise_name in zip(axes_flat, TRACKED_EXERCISES):
        ax.set_facecolor("white")
        color = EXERCISE_COLOR[exercise_name]
        points = timelines[exercise_name]

        if not points:
            ax.text(
                0.5, 0.5,
                "No data found",
                ha="center", va="center",
                transform=ax.transAxes,
                fontsize=11,
                color="#aaaaaa",
            )
            ax.set_title(exercise_name.replace("_", " ").title(), fontsize=12, fontweight="bold", loc="left", pad=8)
            continue

        dates = [p[0] for p in points]
        weights = [p[1] for p in points]

        # Main line
        ax.plot(
            dates, weights,
            color=color,
            linewidth=1.8,
            zorder=2,
        )

        # Scatter dots on top
        ax.scatter(
            dates, weights,
            color=color,
            s=22,
            zorder=3,
            linewidths=0,
        )

        # Light fill below the curve
        ax.fill_between(
            dates, weights,
            alpha=0.12,
            color=color,
            zorder=1,
        )

        # Rolling 5-session average as a guide line (only if enough data)
        if len(weights) >= 5:
            window = 5
            smoothed = np.convolve(weights, np.ones(window) / window, mode="valid")
            smoothed_dates = dates[window - 1:]
            ax.plot(
                smoothed_dates, smoothed,
                color=color,
                linewidth=2.5,
                alpha=0.45,
                linestyle="--",
                zorder=2,
                label="5-session avg",
            )
            ax.legend(fontsize=8, frameon=False, loc="upper left")

        # Axis styling
        ax.set_title(
            exercise_name.replace("_", " ").title(),
            fontsize=12, fontweight="bold", loc="left", pad=8,
        )
        ax.set_ylabel("Weight (kg)", fontsize=9, labelpad=8)
        ax.tick_params(axis="x", rotation=35, labelsize=8)
        ax.tick_params(axis="y", labelsize=9)
        ax.yaxis.grid(True, color="#eeeeee", linewidth=0.8, zorder=0)
        ax.set_axisbelow(True)

        for spine in ["top", "right"]:
            ax.spines[spine].set_visible(False)
        ax.spines["left"].set_color("#cccccc")
        ax.spines["bottom"].set_color("#cccccc")

        ax.yaxis.set_major_formatter(
            ticker.FuncFormatter(lambda v, _: f"{v:.0f} kg")
        )

        # Annotate the personal best
        peak_w = max(weights)
        peak_d = dates[weights.index(peak_w)]
        ax.annotate(
            f"PB {peak_w:.1f} kg",
            xy=(peak_d, peak_w),
            xytext=(8, 8),
            textcoords="offset points",
            fontsize=8,
            color=color,
            fontweight="bold",
        )

    fig.suptitle(
        "Strength Progression Over Time",
        fontsize=14, fontweight="bold", y=1.01,
    )

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Strength progression chart saved to {save_path}")

    return fig
'''

In [ ]:

#prompt


"""
now i want to test an hypothesis such that, 

does more frequent gym sessions correlate with rapid strength increases? 

formulate this into a null hypothesis and alternative hypothesis,
step by step, explain how to identify a critical region such that, we can reject null hypothesis
"""



#output
"""

Null hypothesis (H₀): Training frequency has no significant linear relationship with the rate of strength gain. Any observed correlation is due to random chance.
Alternative hypothesis (H₁): Higher training frequency is positively associated with a faster rate of strength gain.
This is a one-tailed test — we only care about the positive direction (more sessions → faster gains), not the reverse.

Step by step: identifying the critical region
Step 1 — Operationalize the variables
You need two numbers per time window (e.g. per month or per 4-week block):

X (frequency): number of gym sessions in that window
Y (strength gain rate): average weekly increase in the max weight lifted, averaged across the four exercises

Step 2 — Choose the test statistic
Use Pearson's r — the correlation coefficient between X and Y across all windows. It ranges from -1 to +1. The corresponding test statistic is:
t=r1−r2n−2​​
where n is the number of time windows. Under H₀, this t follows a t-distribution with n − 2 degrees of freedom.
Step 3 — Choose a significance level
Use α = 0.05 (5%). Since this is one-tailed (we only reject H₀ for large positive r), the critical region sits entirely in the right tail.
Step 4 — Find the critical value
With n windows and degrees of freedom df = n − 2, the critical value t* is the 95th percentile of the t-distribution: t.ppf(0.95, df=n-2). The critical region is:

Reject H₀ if t_observed > t*

Step 5 — Compute the p-value
The p-value is the probability of seeing a t this large or larger under H₀: p = 1 - t.cdf(t_observed, df=n-2). If p < 0.05, reject H₀.


Step 6 — Draw the conclusion

If t_observed falls in the critical region (p < 0.05): reject H₀, evidence supports that frequency correlates with faster gains
Otherwise: fail to reject H₀, the data does not support the hypothesis at this significance level




'''
hypothesis_test.py
------------------
Tests whether higher training frequency correlates with faster strength gains.

H0: no significant positive linear relationship between frequency and strength gain rate
H1: higher frequency is positively associated with faster strength gain rate

One-tailed Pearson correlation test, alpha = 0.05.

Usage
-----
    from hypothesis_test import test_frequency_vs_strength

    results = test_frequency_vs_strength(data)        # data from load_workout_directory()
    print_hypothesis_results(results)
"""

import numpy as np
from datetime import datetime, date, timedelta
from collections import defaultdict
from scipy import stats


TRACKED_EXERCISES = [
    "bench_press",
    "shoulder_press",
    "leg_extension",
    "lat_pulldown",
]

# Window size in days. 4 weeks gives enough sessions per bucket
# without being so wide that changes within it get lost.
WINDOW_DAYS = 28

ALPHA = 0.05


def _max_weight_for_exercise(workout: dict, exercise_name: str):
    """
    Return the heaviest non-bodyweight load used for an exercise in one workout.
    Returns None if the exercise was not performed.
    """
    for ex in workout["exercises"]:
        if ex["name"] != exercise_name:
            continue
        weights = [
            s["weight_kg"]
            for s in ex["sets"]
            if not s["bodyweight"] and s["weight_kg"] is not None and s["weight_kg"] > 0
        ]
        if weights:
            return max(weights)
    return None


def _build_dated_records(data: dict):
    """
    Convert the raw data dict into a list of records, each being:
        { "date": date, "exercise_name": str, "max_weight_kg": float }

    Only includes workouts with a valid date and exercises from TRACKED_EXERCISES.
    """
    records = []
    for w in data.values():
        if not w["date_str"]:
            continue
        d = datetime.strptime(w["date_str"], "%Y-%m-%d").date()
        for ex_name in TRACKED_EXERCISES:
            mw = _max_weight_for_exercise(w, ex_name)
            if mw is not None:
                records.append({"date": d, "exercise_name": ex_name, "max_weight_kg": mw})
    return sorted(records, key=lambda r: r["date"])


def _make_windows(records: list, window_days: int):
    """
    Slice the full date range into non-overlapping windows of width window_days.

    For each window, compute:
        - session_count    : number of workouts (distinct dates) that fall in it
        - avg_gain_kg_week : average weekly increase in max weight across
                             all tracked exercises that have data in both
                             this window and the previous one

    Returns a list of dicts, one per window that has enough data.
    Each window needs at least one exercise with data in the current
    AND previous window to produce a gain rate estimate.
    """
    if not records:
        return []

    min_date = records[0]["date"]
    max_date = records[-1]["date"]

    # Build windows as (window_start, window_end) pairs
    windows = []
    cursor = min_date
    while cursor <= max_date:
        windows.append((cursor, cursor + timedelta(days=window_days - 1)))
        cursor += timedelta(days=window_days)

    # For each window, group records by exercise
    # window_exercise_max[window_idx][exercise] = max weight seen in that window
    window_exercise_max = defaultdict(lambda: defaultdict(list))
    window_dates = defaultdict(set)

    for r in records:
        for i, (wstart, wend) in enumerate(windows):
            if wstart <= r["date"] <= wend:
                window_exercise_max[i][r["exercise_name"]].append(r["max_weight_kg"])
                window_dates[i].add(r["date"])
                break

    # Collapse lists to max weight per window per exercise
    window_peaks = {}
    for i in range(len(windows)):
        window_peaks[i] = {
            ex: max(weights)
            for ex, weights in window_exercise_max[i].items()
        }

    # Compute gain rate per window (needs comparison with previous window)
    result_windows = []

    for i in range(1, len(windows)):
        prev_peaks = window_peaks.get(i - 1, {})
        curr_peaks = window_peaks.get(i, {})

        # Exercises present in both windows
        common = set(prev_peaks.keys()) & set(curr_peaks.keys())
        if not common:
            continue

        # Gain in kg over one window, converted to kg/week
        gain_rates = []
        for ex in common:
            gain_kg = curr_peaks[ex] - prev_peaks[ex]
            gain_kg_per_week = gain_kg / (window_days / 7)
            gain_rates.append(gain_kg_per_week)

        avg_gain_rate = float(np.mean(gain_rates))

        session_count = len(window_dates.get(i, set()))

        wstart, wend = windows[i]
        result_windows.append({
            "window_label":         f"{wstart} to {wend}",
            "session_count":        session_count,
            "avg_gain_kg_per_week": avg_gain_rate,
            "exercises_tracked":    sorted(common),
        })

    return result_windows


def test_frequency_vs_strength(data: dict, window_days: int = WINDOW_DAYS, alpha: float = ALPHA):
    """
    Test whether training frequency correlates with strength gain rate.

    Parameters
    ----------
    data : dict
        Output of load_workout_directory().
    window_days : int
        Width of each time bucket in days (default 28).
    alpha : float
        Significance level for the one-tailed test (default 0.05).

    Returns
    -------
    dict with keys:
        windows              list of per-window dicts (frequency + gain rate)
        n                    number of windows used
        pearson_r            Pearson correlation coefficient
        t_statistic          computed t value
        t_critical           one-tailed critical value at alpha
        p_value              one-tailed p-value
        degrees_of_freedom   n - 2
        alpha                significance level used
        reject_null          bool, True if we reject H0
        conclusion           plain-English summary string
    """
    records = _build_dated_records(data)
    windows = _make_windows(records, window_days)

    if len(windows) < 4:
        raise ValueError(
            f"Only {len(windows)} usable windows found. "
            "Need at least 4. Try reducing window_days or check that your data "
            "has enough sessions across time."
        )

    freq   = np.array([w["session_count"]        for w in windows], dtype=float)
    gain   = np.array([w["avg_gain_kg_per_week"]  for w in windows], dtype=float)

    n  = len(windows)
    df = n - 2

    # Pearson r
    r, _ = stats.pearsonr(freq, gain)

    # t statistic
    # guard against perfect correlation blowing up the formula
    if abs(r) == 1.0:
        t_stat = np.inf * np.sign(r)
    else:
        t_stat = r * np.sqrt(df / (1 - r ** 2))

    # one-tailed critical value and p-value (right tail)
    t_crit  = stats.t.ppf(1 - alpha, df=df)
    p_value = stats.t.sf(t_stat, df=df)   # sf = 1 - cdf, i.e. right-tail area

    reject = bool(t_stat > t_crit)

    if reject:
        conclusion = (
            f"Reject H0 (p = {p_value:.4f} < alpha = {alpha}). "
            f"There is statistically significant evidence that higher training "
            f"frequency is associated with faster strength gains "
            f"(r = {r:.3f}, t({df}) = {t_stat:.3f})."
        )
    else:
        conclusion = (
            f"Fail to reject H0 (p = {p_value:.4f} >= alpha = {alpha}). "
            f"The data does not provide sufficient evidence that training "
            f"frequency drives faster strength gains "
            f"(r = {r:.3f}, t({df}) = {t_stat:.3f})."
        )

    return {
        "windows":            windows,
        "n":                  n,
        "pearson_r":          float(r),
        "t_statistic":        float(t_stat),
        "t_critical":         float(t_crit),
        "p_value":            float(p_value),
        "degrees_of_freedom": df,
        "alpha":              alpha,
        "reject_null":        reject,
        "conclusion":         conclusion,
    }


def print_hypothesis_results(results: dict):
    """
    Pretty-print the output of test_frequency_vs_strength().
    """
    sep = "-" * 52

    print(sep)
    print("  Frequency vs Strength Gain Rate — Hypothesis Test")
    print(sep)
    print(f"  H0 : no positive correlation between frequency and gain rate")
    print(f"  H1 : higher frequency -> faster strength gains  (one-tailed)")
    print(sep)
    print(f"  Windows analysed     : {results['n']}  (each ~{WINDOW_DAYS} days)")
    print(f"  Degrees of freedom   : {results['degrees_of_freedom']}")
    print(f"  Significance level   : alpha = {results['alpha']}")
    print()
    print(f"  Pearson r            : {results['pearson_r']:+.4f}")
    print(f"  t statistic          : {results['t_statistic']:+.4f}")
    print(f"  t critical (right)   : {results['t_critical']:+.4f}")
    print(f"  p-value (one-tailed) : {results['p_value']:.4f}")
    print()
    verdict = "REJECT H0" if results["reject_null"] else "FAIL TO REJECT H0"
    print(f"  Verdict  >>  {verdict}")
    print()
    print(f"  {results['conclusion']}")
    print(sep)

    print()
    print("  Per-window breakdown:")
    print(f"  {'Window':<26}  {'Sessions':>8}  {'Avg gain (kg/wk)':>16}")
    print(f"  {'-'*26}  {'-'*8}  {'-'*16}")
    for w in results["windows"]:
        print(
            f"  {w['window_label']:<26}  "
            f"{w['session_count']:>8}  "
            f"{w['avg_gain_kg_per_week']:>+16.3f}"
        )
    print(sep)



In [ ]:
#prompt

"""
This project is for examining the datas of my gym session logs, the logs are provided in the .zip file, in hyothesis_test.py file, it already contains 2 tests,
contemplate on a one more different test, integrate it into the hypothesis_test.py file, explain the logic of that test step by step
show how to use it in main

"""



#output


"""
    Test 3 — Session Type vs Total Volume  (one-way ANOVA)
 
    The workout program has three session types: A, B, and C.
    Each type targets a different combination of muscle groups and therefore
    naturally involves different amounts of total load.
 
    The question this test asks:
        Are the mean volumes of the three session types statistically different
        from each other, or is any observed difference just random noise?
 
    H0 : mu_A = mu_B = mu_C
         All three session types produce the same mean total volume.
         (Any difference we see is due to sampling variation.)
 
    H1 : At least one session type has a different mean volume.
         The type label explains a meaningful portion of the variance in volume.
 
    Why one-way ANOVA instead of more Pearson tests?
    ─────────────────────────────────────────────────
    Session type is a *categorical* variable with three levels (A, B, C).
    Pearson correlation requires two continuous variables, so it cannot be
    applied here directly. Running three separate t-tests (A vs B, A vs C,
    B vs C) would inflate the family-wise error rate — the chance of a false
    positive grows with every additional test. One-way ANOVA tests all three
    groups simultaneously with a single F-statistic, keeping alpha honest.
 
    How the F-statistic is built — step by step
    ─────────────────────────────────────────────
    Let k = number of groups (3), N = total sessions across all groups.
 
    1.  Compute the grand mean:
            x̄  = mean of every session's volume, regardless of type.
 
    2.  Between-group sum of squares (SSB):
            SSB = Σ_j  n_j * (x̄_j − x̄)²
        This measures how far each group's mean is from the grand mean,
        weighted by the group's size. Large SSB → the group labels "explain"
        a lot of the total variance.
 
    3.  Within-group sum of squares (SSW):
            SSW = Σ_j  Σ_i  (x_ij − x̄_j)²
        This is the residual variance — how much sessions within the *same*
        type differ from their own group mean. This is the "noise floor."
 
    4.  Mean squares:
            MSB = SSB / (k − 1)         degrees of freedom between groups
            MSW = SSW / (N − k)         degrees of freedom within groups
 
    5.  F statistic:
            F = MSB / MSW
        F > 1 means group means differ more than we'd expect from noise alone.
        Under H0, F follows the F-distribution with (k−1, N−k) degrees of
        freedom.
 
    6.  p-value:
            p = P(F_{k−1, N−k} ≥ F_observed)   [right-tail area]
        If p < alpha we reject H0 and conclude that at least one session type
        has a significantly different mean volume.
 
    Post-hoc (Tukey HSD)
    ─────────────────────
    ANOVA only tells us *that* a difference exists, not *which* pair(s) differ.
    Tukey's Honestly Significant Difference (HSD) test performs all pairwise
    comparisons while controlling the family-wise error rate. Each pair (A–B,
    A–C, B–C) gets its own p-value; we flag pairs where p < alpha.
 
    Assumptions
    ───────────
    • Independence : sessions are recorded on different days → satisfied.
    • Approximate normality : volumes are sums of many weighted sets; by the
      central limit theorem the group means will be approximately normal even
      if individual sessions are somewhat skewed.
    • Homogeneity of variance (homoscedasticity) : we check this with
      Levene's test and report the result alongside the ANOVA output.
      A significant Levene result is a caution flag but does not invalidate
      the ANOVA for large, reasonably balanced groups.
"""
 
 
def _build_type_volume_groups(data: dict) -> dict[str, list[float]]:
    """
    Collect total weighted volume for every session that has:
        - a valid session_type (A, B, or C)
        - at least one weighted set (volume > 0)
 
    Returns
    -------
    dict mapping session_type -> list of per-session volumes (kg × reps)
    """
    groups: dict[str, list[float]] = {"A": [], "B": [], "C": []}
 
    for w in data.values():
        stype = w.get("session_type")
        if stype not in groups:
            continue
 
        vol = 0.0
        for ex in w["exercises"]:
            for s in ex["sets"]:
                if s["bodyweight"] or s["weight_kg"] is None or s["weight_kg"] <= 0:
                    continue
                vol += s["weight_kg"] * sum(s["reps"])
 
        if vol > 0:
            groups[stype].append(vol)
 
    return groups
 
 
def test_session_type_vs_volume(data: dict, alpha: float = ALPHA) -> dict:
    """
    One-way ANOVA: does session type (A / B / C) affect total session volume?
 
    Parameters
    ----------
    data  : dict
        Output of load_workout_directory().
    alpha : float
        Significance level (default 0.05).
 
    Returns
    -------
    dict with keys:
        groups              dict  session_type -> list of volumes
        group_stats         dict  session_type -> {n, mean, std}
        grand_mean          float
        SSB                 float  between-group sum of squares
        SSW                 float  within-group  sum of squares
        df_between          int    k - 1
        df_within           int    N - k
        MSB                 float
        MSW                 float
        F_statistic         float
        F_critical          float  right-tail critical value at alpha
        p_value             float
        alpha               float
        reject_null         bool
        conclusion          str    plain-English summary
        levene_statistic    float  Levene's test for equal variances
        levene_p            float
        levene_warning      str | None
        tukey_results       list of dicts  [{pair, mean_diff, p_tukey, significant}]
    """
    groups = _build_type_volume_groups(data)
 
    # Require at least 2 sessions per group to proceed
    for stype, vols in groups.items():
        if len(vols) < 2:
            raise ValueError(
                f"Session type '{stype}' has only {len(vols)} usable session(s). "
                "Need at least 2 per group to run ANOVA."
            )
 
    k = len(groups)                          # number of groups (3)
    N = sum(len(v) for v in groups.values()) # total sessions
 
    all_vols = np.concatenate([np.array(v) for v in groups.values()])
    grand_mean = float(np.mean(all_vols))
 
    # ── Step 2-3: Sum of squares ──────────────────────────────────────
    SSB = 0.0
    SSW = 0.0
    group_stats = {}
 
    for stype, vols in groups.items():
        arr = np.array(vols, dtype=float)
        n_j = len(arr)
        mean_j = float(np.mean(arr))
        std_j  = float(np.std(arr, ddof=1))
 
        group_stats[stype] = {"n": n_j, "mean": mean_j, "std": std_j}
 
        SSB += n_j * (mean_j - grand_mean) ** 2
        SSW += float(np.sum((arr - mean_j) ** 2))
 
    # ── Step 4-5: Mean squares and F ─────────────────────────────────
    df_between = k - 1
    df_within  = N - k
 
    MSB = SSB / df_between
    MSW = SSW / df_within
    F   = MSB / MSW
 
    # ── Step 6: p-value and critical value ───────────────────────────
    F_crit  = stats.f.ppf(1 - alpha, dfn=df_between, dfd=df_within)
    p_value = float(stats.f.sf(F, dfn=df_between, dfd=df_within))
 
    reject = bool(p_value < alpha)
 
    if reject:
        conclusion = (
            f"Reject H0 (p = {p_value:.4f} < alpha = {alpha}). "
            f"At least one session type has a significantly different mean volume "
            f"(F({df_between}, {df_within}) = {F:.3f}). "
            f"Session type explains meaningful variance in total session load."
        )
    else:
        conclusion = (
            f"Fail to reject H0 (p = {p_value:.4f} >= alpha = {alpha}). "
            f"No significant difference in mean volume across session types "
            f"(F({df_between}, {df_within}) = {F:.3f}). "
            f"Observed differences are within the range of random variation."
        )
 
    # ── Levene's test for homogeneity of variance ────────────────────
    lev_stat, lev_p = stats.levene(*[np.array(v) for v in groups.values()])
    levene_warning = None
    if lev_p < alpha:
        levene_warning = (
            f"Levene's test is significant (p = {lev_p:.4f}), suggesting "
            "unequal variances across groups. Interpret the ANOVA result with "
            "caution; the F-test is less reliable under heteroscedasticity."
        )
 
    # ── Tukey HSD post-hoc pairwise comparisons ──────────────────────
    # scipy pairwise_tukeyhsd expects a flat array of values + group labels
    from scipy.stats import tukey_hsd
    type_labels = sorted(groups.keys())           # ["A", "B", "C"]
    arrays      = [np.array(groups[t]) for t in type_labels]
 
    tukey_out   = tukey_hsd(*arrays)
    tukey_results = []
    for i in range(len(type_labels)):
        for j in range(i + 1, len(type_labels)):
            t_i = type_labels[i]
            t_j = type_labels[j]
            mean_diff = group_stats[t_i]["mean"] - group_stats[t_j]["mean"]
            p_tukey   = float(tukey_out.pvalue[i][j])
            tukey_results.append({
                "pair":        f"{t_i} vs {t_j}",
                "mean_diff_kg": mean_diff,
                "p_tukey":     p_tukey,
                "significant": p_tukey < alpha,
            })
 
    return {
        "groups":           groups,
        "group_stats":      group_stats,
        "grand_mean":       grand_mean,
        "SSB":              SSB,
        "SSW":              SSW,
        "df_between":       df_between,
        "df_within":        df_within,
        "MSB":              MSB,
        "MSW":              MSW,
        "F_statistic":      float(F),
        "F_critical":       float(F_crit),
        "p_value":          p_value,
        "alpha":            alpha,
        "reject_null":      reject,
        "conclusion":       conclusion,
        "levene_statistic": float(lev_stat),
        "levene_p":         float(lev_p),
        "levene_warning":   levene_warning,
        "tukey_results":    tukey_results,
    }
 
 
def print_session_type_results(results: dict):
    """
    Pretty-print the output of test_session_type_vs_volume().
    """
    sep = "-" * 60
    alpha = results["alpha"]
 
    print(sep)
    print("  Session Type vs Total Volume — One-Way ANOVA")
    print(sep)
    print("  H0 : mu_A = mu_B = mu_C  (all types have equal mean volume)")
    print("  H1 : at least one session type has a different mean volume")
    print(sep)
 
    # Group summary table
    print(f"  {'Type':<6}  {'n':>5}  {'Mean (kg)':>10}  {'Std (kg)':>10}")
    print(f"  {'-'*6}  {'-'*5}  {'-'*10}  {'-'*10}")
    for stype in sorted(results["group_stats"]):
        gs = results["group_stats"][stype]
        print(
            f"  {stype:<6}  {gs['n']:>5}  "
            f"{gs['mean']:>10.1f}  {gs['std']:>10.1f}"
        )
    print(f"  {'Grand mean':<14}  {results['grand_mean']:>10.1f}")
    print()
 
    # ANOVA table
    print(f"  {'Source':<16}  {'SS':>12}  {'df':>5}  {'MS':>12}  {'F':>8}")
    print(f"  {'-'*16}  {'-'*12}  {'-'*5}  {'-'*12}  {'-'*8}")
    print(
        f"  {'Between groups':<16}  {results['SSB']:>12.1f}  "
        f"{results['df_between']:>5}  {results['MSB']:>12.1f}  "
        f"{results['F_statistic']:>8.3f}"
    )
    print(
        f"  {'Within groups':<16}  {results['SSW']:>12.1f}  "
        f"{results['df_within']:>5}  {results['MSW']:>12.1f}  "
        f"{'':>8}"
    )
    print()
    print(f"  F critical (alpha={alpha})  : {results['F_critical']:.4f}")
    print(f"  p-value                    : {results['p_value']:.4f}")
    print()
 
    verdict = "REJECT H0" if results["reject_null"] else "FAIL TO REJECT H0"
    print(f"  Verdict  >>  {verdict}")
    print()
    print(f"  {results['conclusion']}")
    print()
 
    # Levene's test
    lev_flag = "  ⚠" if results["levene_warning"] else "  ✓"
    print(
        f"{lev_flag} Levene's test: W = {results['levene_statistic']:.3f}, "
        f"p = {results['levene_p']:.4f}"
    )
    if results["levene_warning"]:
        print(f"    {results['levene_warning']}")
    print()
 
    # Tukey HSD
    print("  Tukey HSD pairwise comparisons:")
    print(f"  {'Pair':<12}  {'Mean diff (kg)':>14}  {'p (Tukey)':>10}  {'Significant?':>12}")
    print(f"  {'-'*12}  {'-'*14}  {'-'*10}  {'-'*12}")
    for tr in results["tukey_results"]:
        sig = "YES *" if tr["significant"] else "no"
        print(
            f"  {tr['pair']:<12}  {tr['mean_diff_kg']:>+14.1f}  "
            f"{tr['p_tukey']:>10.4f}  {sig:>12}"
        )
    print(sep)


"""

In [ ]:
#prompt
"""
i want to create a heatmap that show my gym sessions as points on the calendar,
achieve this using matplotlib,
create a seperate function
clearly show the input and output of the function
"""



#output
'''

import calendar
from datetime import datetime, date
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np


# ─────────────────────────────────────────────
# INPUT:  data        → dict returned by load_workout_directory()
#                       keys: "W-NNN-X", values: parsed workout dicts
#         save_path   → str | None — if given, saves PNG to that path
#
# OUTPUT: matplotlib.figure.Figure
#         A calendar heatmap where each day cell is coloured by
#         session type(s) logged that day (A=blue, B=orange, C=green).
#         Cells with no session are empty (light grey).
# ─────────────────────────────────────────────
def plot_workout_heatmap(data: dict, save_path: str | None = None):
    """
    Render a GitHub-style calendar heatmap of all workout sessions.

    Parameters
    ----------
    data : dict
        Output of load_workout_directory(). Each value must contain
        'date_str' (ISO "YYYY-MM-DD") and 'session_type' ("A", "B", or "C").
    save_path : str or None
        File path to save the figure (e.g. "heatmap.png").
        If None the figure is shown interactively.

    Returns
    -------
    matplotlib.figure.Figure
        The rendered heatmap figure.
    """

    # ── 1. Collect (date, session_type) pairs ──────────────────────────
    sessions: list[tuple[date, str]] = []
    for w in data.values():
        if w["date_str"] and w["session_type"]:
            d = datetime.strptime(w["date_str"], "%Y-%m-%d").date()
            sessions.append((d, w["session_type"]))

    if not sessions:
        raise ValueError("No sessions with valid dates found in data.")

    # ── 2. Determine date range (full years) ──────────────────────────
    min_date = min(d for d, _ in sessions)
    max_date = max(d for d, _ in sessions)
    start = date(min_date.year, 1, 1)
    end   = date(max_date.year, 12, 31)

    # ── 3. Build lookup: date → list of session types ─────────────────
    from collections import defaultdict
    day_sessions: dict[date, list[str]] = defaultdict(list)
    for d, stype in sessions:
        day_sessions[d].append(stype)

    # ── 4. Color map  ─────────────────────────────────────────────────
    TYPE_COLOR = {"A": "#4C9BE8", "B": "#F5A623", "C": "#57C26A"}
    EMPTY_COLOR = "#EEEEEE"

    def cell_color(day: date) -> str:
        types = day_sessions.get(day, [])
        if not types:
            return EMPTY_COLOR
        # If multiple types logged same day, pick A > B > C priority
        for t in ("A", "B", "C"):
            if t in types:
                return TYPE_COLOR[t]
        return EMPTY_COLOR

    # ── 5. Layout: one row per year, columns = week of year ───────────
    years = list(range(start.year, end.year + 1))
    n_years = len(years)

    fig, axes = plt.subplots(
        n_years, 1,
        figsize=(20, 2.2 * n_years + 0.8),
        facecolor="white"
    )
    if n_years == 1:
        axes = [axes]

    DAYS   = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    for ax, year in zip(axes, years):
        ax.set_facecolor("white")
        ax.set_xlim(-0.5, 53.5)
        ax.set_ylim(-0.5, 7.5)
        ax.invert_yaxis()
        ax.set_yticks(range(7))
        ax.set_yticklabels(DAYS, fontsize=8)
        ax.tick_params(left=False, bottom=False)
        for spine in ax.spines.values():
            spine.set_visible(False)

        ax.set_title(str(year), loc="left", fontsize=11, fontweight="bold", pad=6)

        # Month label positions
        month_week_starts = {}
        for month in range(1, 13):
            first = date(year, month, 1)
            wk = first.isocalendar()[1]
            # week 1 of following year wraps to 0
            if month == 1:
                wk = 0
            month_week_starts[month] = wk

        for month, wk in month_week_starts.items():
            ax.text(wk, -0.6, MONTHS[month - 1],
                    fontsize=7.5, color="#666666", va="bottom")

        # Draw squares
        for month in range(1, 13):
            _, n_days = calendar.monthrange(year, month)
            for day_num in range(1, n_days + 1):
                d = date(year, month, day_num)
                iso = d.isocalendar()          # (year, week, weekday)
                week_col = iso[1] - 1          # 0-indexed column
                if month == 1 and iso[1] > 50: # last week of prev year
                    week_col = 0
                dow = d.weekday()              # 0=Mon … 6=Sun
                color = cell_color(d)
                rect = mpatches.FancyBboxPatch(
                    (week_col + 0.07, dow + 0.07),
                    0.86, 0.86,
                    boxstyle="round,pad=0.05",
                    linewidth=0,
                    facecolor=color
                )
                ax.add_patch(rect)

    # ── 6. Legend ─────────────────────────────────────────────────────
    legend_handles = [
        mpatches.Patch(color=c, label=f"Session {t}")
        for t, c in TYPE_COLOR.items()
    ] + [mpatches.Patch(color=EMPTY_COLOR, label="No session")]

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=4,
        fontsize=9,
        frameon=False,
        bbox_to_anchor=(0.5, 0.0)
    )

    fig.suptitle("Workout Calendar Heatmap", fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout(rect=[0, 0.04, 1, 1])

    # ── 7. Save or show ───────────────────────────────────────────────
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Heatmap saved → {save_path}")

    return fig


'''